Этот нотбук реализует концепцию **Active Learning (активного обучения)** для оптимизации ручной разметки:
1. Загрузка классифицированного датасета `data_classified.csv`.
2. Загрузка файла ручной разметки `manual_labeling_sample.csv` (если файл отсутствует, он создается автоматически).
3. Идентификация неразмеченных постов по их уникальному ключу `id`.
4. Выборка **самых неопределенных постов** (где `signed_score` наиболее близок к `0.0`).
5. Интерактивный интерфейс разметки для пользователя с поддержкой быстрых клавиш.
6. Сохранение обновленного набора размеченных данных без риска возникновения дубликатов.

In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import clear_output, display

# Настройки путей к файлам
DATA_DIR = "data"
CLASSIFIED_DATA_PATH = os.path.join(DATA_DIR, "data_classified.csv")
LABELED_DATA_PATH = os.path.join(DATA_DIR, "manual_labeling_sample.csv")

In [ ]:
# 1. Загрузка классифицированных данных
if not os.path.exists(CLASSIFIED_DATA_PATH):
    raise FileNotFoundError(
        f"Файл классифицированных данных {CLASSIFIED_DATA_PATH} не найден. "
        "Сначала запустите Notebook 3 для генерации прогнозов."
    )

df_classified = pd.read_csv(CLASSIFIED_DATA_PATH)
print(f"Успешно загружено классифицированных постов: {len(df_classified)}")

# 2. Загрузка или инициализация файла ручной разметки
# Задаем 'id' как уникальный ключ для сопоставления
if os.path.exists(LABELED_DATA_PATH):
    df_labeled = pd.read_csv(LABELED_DATA_PATH)
    # Удаляем дубликаты по id на случай непредвиденных сбоев
    df_labeled = df_labeled.drop_duplicates(subset=["id"]).copy()
    print(f"Успешно загружено существующих размеченных записей: {len(df_labeled)}")
else:
    # Инициализируем пустой DataFrame, сохраняя обратную совместимость с Notebook 2
    df_labeled = pd.DataFrame(columns=["id", "text", "preliminary_label"])
    df_labeled.to_csv(LABELED_DATA_PATH, index=False, encoding="utf-8-sig")
    print("Файл ручной разметки не найден. Создан новый пустой файл.")

In [ ]:
# Определение количества постов для разметки в текущей сессии
N_TO_LABEL = 10 

# Поиск ID постов, которых еще нет в ручной разметке
unlabeled_mask = ~df_classified["id"].isin(df_labeled["id"])
df_unlabeled = df_classified[unlabeled_mask].copy()

if len(df_unlabeled) == 0:
    print("Все посты из базы данных уже были размечены!")
else:
    # Active Learning: Сортировка по степени неуверенности модели.
    # Модель наиболее не уверена в постах, чей signed_score максимально близок к 0.0.
    df_unlabeled["uncertainty"] = df_unlabeled["signed_score"].abs()
    
    # Выборка ТОП-N наиболее неуверенных кандидатов
    df_candidates = df_unlabeled.sort_values(by="uncertainty").head(N_TO_LABEL)
    
    print(f"Всего неразмеченных постов в базе: {len(df_unlabeled)}")
    print(f"Отобрано наиболее неуверенных кандидатов для разметки: {len(df_candidates)}")
    display(df_candidates[["id", "channel", "signed_score", "confidence_tier"]])

In [ ]:
# Быстрые клавиши для удобства разметки
LABEL_MAPPING = {
    "v": "vacancy", "vacancy": "vacancy",
    "r": "resume", "resume": "resume",
    "t": "trash", "trash": "trash"
}

# Список для временного хранения новых меток
new_labels = []

if len(df_unlabeled) > 0:
    print("Запуск сессии интерактивной разметки...")
    
    for idx, row in enumerate(df_candidates.itertuples(), 1):
        clear_output(wait=True)
        
        # Визуальное оформление шапки
        print("=" * 80)
        print(f"Пост {idx} из {len(df_candidates)} | ID: {row.id} | Канал: {row.channel}")
        print(f"Текущие оценки модели -> P(Vacancy): {row.p_vacancy:.4f} | Signed Score: {row.signed_score:.4f}")
        print("=" * 80)
        print(f"\n{row.original_text}\n")
        print("=" * 80)
        print("Команды разметки:")
        print("  v - Вакансия (vacancy)  |  r - Резюме (resume)  |  t - Мусор (trash)")
        print("  s - Пропустить (skip)   |  e - Сохранить и выйти (exit)")
        print("=" * 80)
        
        while True:
            user_input = input("Введите команду: ").strip().lower()
            
            if user_input in ["e", "exit"]:
                print("\nСессия прервана пользователем. Переход к сохранению результатов...")
                break
            elif user_input in ["s", "skip"]:
                print("Пост пропущен.")
                break
            elif user_input in LABEL_MAPPING:
                mapped_label = LABEL_MAPPING[user_input]
                new_labels.append({
                    "id": row.id,
                    "text": row.original_text,
                    "preliminary_label": mapped_label
                })
                print(f"Принято: {mapped_label}")
                break
            else:
                print("Ошибка: неверная команда. Пожалуйста, введите v, r, t, s или e.")
        
        # Если пользователь выбрал выход, останавливаем цикл постов
        if user_input in ["e", "exit"]:
            break

    clear_output(wait=True)
    print("Сессия разметки завершена.")
    
    # 3. Интеграция и сохранение результатов
    if len(new_labels) > 0:
        df_new = pd.DataFrame(new_labels)
        
        # Объединяем старые и новые данные
        df_labeled_updated = pd.concat([df_labeled, df_new], ignore_index=True)
        
        # Гарантируем отсутствие дубликатов по уникальному ключу id
        # При возникновении конфликта сохраняется первая (оригинальная) запись
        df_labeled_updated = df_labeled_updated.drop_duplicates(subset=["id"], keep="first").copy()
        
        # Сохранение в файл
        df_labeled_updated.to_csv(LABELED_DATA_PATH, index=False, encoding="utf-8-sig")
        
        print(f"Успешно сохранено новых записей: {len(new_labels)}")
        print(f"Общий объем размеченного датасета: {len(df_labeled_updated)}")
    else:
        print("Новых записей для сохранения нет.")